In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from enum import IntEnum
from functools import lru_cache

### Загрузка данных

In [2]:
df = pd.read_csv('../results_new.csv', comment='#')
print(df.shape)
df.head(3)

(24240600, 22)


,delta1,delta2,eps,coupling_type,x0,y0,x1,y1,x2,y2,...,xf1,yf1,xf2,yf2,L,A,P,s01,s02,s12
0,-0.5,-0.5,0.01,0,2.982814,-0.260313,1.504800,2.480039,0.312471,-1.414831,...,0.386961,0.283261,-0.456529,0.198688,1.287644,0.348388,0.510182,0.500903,0.500929,2.622361e-05
1,-0.5,-0.5,0.01,0,-0.313921,-1.531953,-2.891272,-2.336488,-0.765504,1.189760,...,-0.383866,0.238028,0.393579,-0.224217,0.379705,0.148777,0.333371,0.500779,0.500778,3.386947e-07
2,-0.5,-0.5,0.01,0,-1.679338,-1.358138,-1.477282,0.111581,-1.521640,1.782043,...,0.188237,0.335113,0.530894,-0.155000,1.034098,0.295733,0.476975,0.500893,0.500861,3.234697e-05


### Округление точности

In [3]:
df['delta1'] = df['delta1'].round(6)
df['delta2'] = df['delta2'].round(6)
df['eps'] = df['eps'].round(6)

In [ ]:
COUPLING_NAMES = ['Inertial', 'InertialNorm', 'Dissipative', 'DissipativeNorm']
epsilons = sorted(df['eps'].unique())

class CouplingType(IntEnum):
    Inertial = 0
    InertialNorm = 1
    Dissipative = 2
    DissipativeNorm = 3

THETA = {ct: 0.025 for ct in CouplingType}

# Код строится только по двум реально связанным парам (0↔1, 0↔2 — рёбра звезды).
# s12 не независим: по тождеству (φ0-φ1)=(φ0-φ2)-(φ1-φ2) и линейности МНК-наклона
# A01 = A02 - A12 точно, а после std::abs() в C++ третий бит превращается в
# артефакт (|A02-A12| может быть суммой модулей при разных знаках) — поэтому
# в код режима его не включаем, s12 остаётся только диагностикой.
REGIME_LABELS = ['нет синхр.', '0↔1', '0↔2', 'полная (0↔1, 0↔2)']
REGIME_COLORS = ['#d3d3d3', '#4477aa', '#66ccee', '#222222']
MULTI_BORDER_COLOR = '#ffffff'
MULTI_BORDER_WIDTH = 2

print('epsilons:', epsilons)
print('coupling types:', sorted(df['coupling_type'].unique()))
print('runs per point:', df.groupby(['delta1', 'delta2', 'eps', 'coupling_type']).size().unique())

In [ ]:
theta_col = df['coupling_type'].map(THETA).to_numpy()
df['code'] = (((df['s01'].to_numpy() < theta_col).astype(np.int8))
              | ((df['s02'].to_numpy() < theta_col).astype(np.int8) << 1))

P_INPHASE = 0.9               # порог P, выше которого запуск считаем синфазным
df['p_hi'] = df['P'] > P_INPHASE

@lru_cache(maxsize=None)
def sub_with_code(eps):
    return df[df['eps'] == eps]

# Один проход агрегации по точкам сетки (delta1, delta2, eps, тип связи) —
# дальше все карты слайсят PTS, а не сканируют df заново.
KEYS = ['delta1', 'delta2', 'eps', 'coupling_type']
PTS = df.groupby(KEYS).agg(
    L_max=('L', 'max'), L_min=('L', 'min'),
    A_max=('A', 'max'), A_min=('A', 'min'),
    P_max=('P', 'max'), P_min=('P', 'min'),
    inphase_frac=('p_hi', 'mean'),
    n_codes=('code', 'nunique'),
).reset_index()
for _v in ('L', 'A', 'P'):
    PTS[f'{_v}_spread'] = PTS[f'{_v}_max'] - PTS[f'{_v}_min']
PTS['multi'] = PTS['n_codes'] > 1

# Доминирующий код, чистота моды и разбивка по кодам для ховера.
_counts = df.groupby(KEYS + ['code']).size().rename('n').reset_index()
_dom = (_counts.sort_values('n').drop_duplicates(KEYS, keep='last')
        .rename(columns={'code': 'dominant_code'})[KEYS + ['dominant_code']])
PTS = PTS.merge(_dom, on=KEYS)

_pt = _counts.groupby(KEYS)['n'].agg(dom_n='max', tot_n='sum')
_pt['agree_frac'] = _pt['dom_n'] / _pt['tot_n']
PTS = PTS.merge(_pt[['agree_frac']].reset_index(), on=KEYS)

_counts['lbl'] = np.array(REGIME_LABELS)[_counts['code'].to_numpy()] + ': ' + _counts['n'].astype(str)
_brk = (_counts.sort_values('n', ascending=False).groupby(KEYS)['lbl']
        .agg('<br>'.join).rename('breakdown').reset_index())
PTS = PTS.merge(_brk, on=KEYS)

def discrete_colorscale(colors):
    n = len(colors)
    return [pt for i, c in enumerate(colors) for pt in ([i / n, c], [(i + 1) / n, c])]

REGIME_SCALE = discrete_colorscale(REGIME_COLORS)

def _eps_step(name):
    return dict(method='animate', label=name,
                args=[[name], dict(mode='immediate', frame=dict(duration=0, redraw=True),
                                   transition=dict(duration=0))])


def add_eps_slider(fig, frames, prefix='ε = ', pad_t=60):
    fig.frames = list(frames)
    fig.update_layout(sliders=[dict(active=0, pad={'t': pad_t},
                                    currentvalue={'prefix': prefix},
                                    steps=[_eps_step(f.name) for f in fig.frames])])
    return fig


def add_eps_coupling_sliders(fig, frames, group_size, coupling_names):
    fig.frames = list(frames)
    total = len(coupling_names) * group_size
    coupling_steps = [dict(method='restyle', label=name,
                           args=[{'visible': [ci * group_size <= i < (ci + 1) * group_size
                                              for i in range(total)]}])
                      for ci, name in enumerate(coupling_names)]
    fig.update_layout(sliders=[
        dict(active=0, yanchor='top', y=0, pad={'t': 40},
             currentvalue={'prefix': 'ε = '}, steps=[_eps_step(f.name) for f in fig.frames]),
        dict(active=0, yanchor='top', y=0, pad={'t': 110},
             currentvalue={'prefix': 'тип связи: '}, steps=coupling_steps),
    ])
    return fig

## Подбор порога θ

### Распределение наклонов разности фаз

In [6]:
def kde_long():
    x_grid = np.linspace(0, 0.2, 400)
    rng = np.random.default_rng(0)
    out = []
    for (e, c), g in df.groupby(['eps', 'coupling_type']):
        s = np.abs(np.concatenate([g['s01'].values, g['s02'].values, g['s12'].values]))
        if s.size > 50000:
            s = rng.choice(s, 50000, replace=False)
        y = gaussian_kde(s, bw_method=0.03)(x_grid)
        out.append(pd.DataFrame({'eps': e, 'coupling': COUPLING_NAMES[c],
                                 '|slope|': x_grid, 'density': y}))
    return pd.concat(out, ignore_index=True)


kdf = kde_long()
fig = px.line(kdf, x='|slope|', y='density',
              facet_col='coupling', facet_col_wrap=2,
              animation_frame='eps', log_y=True, height=800, width=800,
              title='Плотность |slope| (KDE)')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
dens = kdf['density']
fig.update_yaxes(range=[np.log10(dens[dens > 0].min()), np.log10(dens.max() * 1.2)])
fig.layout.updatemenus = []
fig.show()

In [ ]:
pd.Series({CouplingType(ct).name: THETA[CouplingType(ct)] for ct in range(4)}, name='θ').to_frame()

### Карта наклона и устойчивость к θ

`min|slope|` по н.у. (плато ≈ 0 — захват) + контуры θ ∈ {0.01…0.05}. Контуры сбиты в тонкую полосу ⇒ граница захвата не зависит от выбора θ.

In [ ]:
SLOPE_MIN = df.groupby(KEYS)[['s01', 's02', 's12']].min().reset_index()
_PAIRS = [('s01', '0–1'), ('s02', '0–2'), ('s12', '1–2')]


def slope_traces(eps):
    a = SLOPE_MIN[SLOPE_MIN['eps'] == eps]
    traces = []
    for ct in range(4):
        d = a[a['coupling_type'] == ct]
        for pi, (col, _) in enumerate(_PAIRS):
            piv = d.pivot(index='delta1', columns='delta2', values=col)
            traces.append(go.Heatmap(
                z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
                colorscale='Viridis', zmin=0, zmax=0.06, showscale=(pi == 0),
                colorbar=dict(title='min|slope|', x=1.02, len=0.9, thickness=12),
                hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>%{z:.4f}<extra></extra>'))
            traces.append(go.Contour(
                z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
                contours=dict(start=0.01, end=0.05, size=0.01, coloring='lines'),
                line=dict(width=1, color='white'), showscale=False, hoverinfo='skip'))
    return traces


fig = make_subplots(rows=1, cols=3, subplot_titles=[lbl for _, lbl in _PAIRS], horizontal_spacing=0.06)
for idx, tr in enumerate(slope_traces(epsilons[0])):
    tr.visible = (idx // 6 == 0)
    panel = (idx % 6) // 2
    fig.add_trace(tr, row=1, col=panel + 1)

frames = [go.Frame(name=str(e), data=slope_traces(e)) for e in epsilons]
fig.update_layout(height=450, width=1300, margin=dict(b=140), title='min|slope| и контуры θ')
add_eps_coupling_sliders(fig, frames, group_size=6, coupling_names=COUPLING_NAMES)
fig.show()

## Карты режимов синхронизации

In [ ]:
all_agg = {e: PTS[PTS['eps'] == e] for e in epsilons}

In [ ]:
_LBL_ARR = np.array(REGIME_LABELS)


def regime_traces(eps):
    a = all_agg[eps]
    traces = []
    for ct in range(4):
        d = a[a['coupling_type'] == ct]
        dom = d.pivot(index='delta1', columns='delta2', values='dominant_code')
        brk = d.pivot(index='delta1', columns='delta2', values='breakdown')
        label = _LBL_ARR[dom.values.astype(int)]
        custom = np.dstack([label, brk.values])
        traces.append(go.Heatmap(
            z=dom.values, x=dom.columns.tolist(), y=dom.index.tolist(),
            customdata=custom, colorscale=REGIME_SCALE, zmin=-0.5, zmax=3.5, showscale=False,
            hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>мода: %{customdata[0]}'
                          '<br>%{customdata[1]}<extra></extra>'))
    return traces


fig = go.Figure()
for idx, tr in enumerate(regime_traces(epsilons[0])):
    tr.visible = (idx == 0)
    fig.add_trace(tr)

frames = [go.Frame(name=str(e), data=regime_traces(e)) for e in epsilons]

fig.update_layout(
    height=800, width=800, autosize=True, plot_bgcolor='white',
    margin=dict(b=140), xaxis_title='δ₂', yaxis_title='δ₁',
    title='Режимы синхронизации (мода; ховер — разбивка по н.у.)'
)
add_eps_coupling_sliders(fig, frames, group_size=1, coupling_names=COUPLING_NAMES)
fig.show()

## Целевые функции

In [ ]:
_GOALS = ['L', 'A', 'P']
GOAL_LO = {g: float(PTS[f'{g}_min'].min()) for g in _GOALS}
GOAL_HI = {g: float(PTS[f'{g}_max'].max()) for g in _GOALS}
GOAL_SPREAD_HI = {g: float(PTS[f'{g}_spread'].max()) for g in _GOALS}


def goal_heatmap(piv, scale, zmin, zmax):
    return go.Heatmap(z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
                      colorscale=scale, zmin=zmin, zmax=zmax, showscale=False,
                      hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>%{z:.3f}<extra></extra>')


def goal_traces(eps):
    a = PTS[PTS['eps'] == eps]
    traces = []
    for ct in range(4):
        d = a[a['coupling_type'] == ct]
        for g in _GOALS:
            traces.append(goal_heatmap(d.pivot(index='delta1', columns='delta2', values=f'{g}_max'),
                                       'YlOrRd', GOAL_LO[g], GOAL_HI[g]))
        for g in _GOALS:
            traces.append(goal_heatmap(d.pivot(index='delta1', columns='delta2', values=f'{g}_spread'),
                                       'Magma', 0.0, GOAL_SPREAD_HI[g]))
    return traces


fig = make_subplots(rows=2, cols=3, column_titles=_GOALS, row_titles=['max', 'Δ'],
                    horizontal_spacing=0.06, vertical_spacing=0.10)
for idx, tr in enumerate(goal_traces(epsilons[0])):
    tr.visible = (idx // 6 == 0)
    i = idx % 6
    fig.add_trace(tr, row=i // 3 + 1, col=i % 3 + 1)

frames = [go.Frame(name=str(e), data=goal_traces(e)) for e in epsilons]
fig.update_layout(height=800, width=1200, margin=dict(b=140), title='Целевые функции: max и разброс Δ по н.у.')
add_eps_coupling_sliders(fig, frames, group_size=6, coupling_names=COUPLING_NAMES)
fig.show()

In [ ]:
A_EDGES = np.linspace(df['A'].min(), df['A'].max(), 61)
P_EDGES = np.linspace(df['P'].min(), df['P'].max(), 61)
_AC = 0.5 * (A_EDGES[:-1] + A_EDGES[1:])
_PC = 0.5 * (P_EDGES[:-1] + P_EDGES[1:])

_PA_HIST = {}
for (e, ct), g in df.groupby(['eps', 'coupling_type']):
    h, _, _ = np.histogram2d(g['A'].values, g['P'].values, bins=[A_EDGES, P_EDGES])
    _PA_HIST[(e, ct)] = np.log1p(h.T)


def pa_density(eps):
    return [go.Heatmap(z=_PA_HIST[(eps, ct)], x=_AC, y=_PC,
                       colorscale='Blues', showscale=False,
                       hovertemplate='A=%{x:.3f} P=%{y:.3f}<br>log(1+n)=%{z:.2f}<extra></extra>')
            for ct in range(4)]


fig = make_subplots(rows=1, cols=4, subplot_titles=COUPLING_NAMES,
                    horizontal_spacing=0.04, shared_yaxes=True)
for i, tr in enumerate(pa_density(epsilons[0])):
    fig.add_trace(tr, row=1, col=i + 1)

frames = [go.Frame(name=str(e), data=pa_density(e)) for e in epsilons]
fig.update_xaxes(title_text='A')
fig.update_yaxes(title_text='P', col=1)
fig.update_layout(height=400, width=1300, title='P vs A (log-плотность по запускам)')
add_eps_slider(fig, frames)
fig.show()

In [ ]:
def _corrs(g):
    return pd.Series({'ρ_Pearson': g['A'].corr(g['P']),
                      'ρ_Spearman': g['A'].corr(g['P'], method='spearman')})


PA_CORR = df.groupby(['eps', 'coupling_type'])[['A', 'P']].apply(_corrs).reset_index()
PA_CORR['coupling'] = PA_CORR['coupling_type'].map(dict(enumerate(COUPLING_NAMES)))

_m = PA_CORR.pivot(index='eps', columns='coupling', values='ρ_Spearman')[COUPLING_NAMES]
fig = go.Figure(go.Heatmap(
    z=_m.values, x=_m.columns.tolist(), y=[f'{e:.2f}' for e in _m.index],
    text=_m.values, texttemplate='%{text:.3f}',
    colorscale='RdYlGn', zmin=float(_m.values.min()), zmax=1.0,
    colorbar=dict(title='ρ_Spearman')))
fig.update_layout(height=420, width=640, title='ρ_Spearman(A, P) по ε и типу связи',
                  xaxis_title='тип связи', yaxis_title='ε')
fig.show()

## Синфазность

- **max P** — максимум $P$ по начальным условиям (есть ли синфазный режим);
- **доля P>0.9** — доля начальных условий с синфазным режимом (насколько он типичен).

In [ ]:
P_LO = float(PTS['P_min'].min())


def inphase_heatmap(piv, scale, zmin, zmax, cbar_title, cbar_x):
    return go.Heatmap(z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
                      colorscale=scale, zmin=zmin, zmax=zmax, showscale=True,
                      colorbar=dict(title=cbar_title, x=cbar_x, len=0.9, thickness=12),
                      hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>%{z:.3f}<extra></extra>')


def inphase_traces(eps):
    a = PTS[PTS['eps'] == eps]
    traces = []
    for ct in range(4):
        d = a[a['coupling_type'] == ct]
        pmax = d.pivot(index='delta1', columns='delta2', values='P_max')
        frac = d.pivot(index='delta1', columns='delta2', values='inphase_frac')
        traces.append(inphase_heatmap(pmax, 'Viridis', P_LO, 1.0, 'max P', 0.44))
        traces.append(inphase_heatmap(frac, 'Viridis', 0.0, 1.0, f'доля P>{P_INPHASE}', 1.0))
    return traces


fig = make_subplots(rows=1, cols=2, subplot_titles=['max P', f'доля P>{P_INPHASE}'],
                    horizontal_spacing=0.18)
for idx, tr in enumerate(inphase_traces(epsilons[0])):
    tr.visible = (idx // 2 == 0)
    fig.add_trace(tr, row=1, col=idx % 2 + 1)

frames = [go.Frame(name=str(e), data=inphase_traces(e)) for e in epsilons]
fig.update_layout(height=600, width=1200, margin=dict(b=140), title='Синфазность: max P и доля синфазных н.у.')
add_eps_coupling_sliders(fig, frames, group_size=2, coupling_names=COUPLING_NAMES)
fig.show()
fig.write_html('../html/inphase.html')

## Исследование $\Delta L$, $\Delta A$, $\Delta P$ как критериев мультистабильности

In [ ]:
# Мультистабильность (θ): доля н.у. НЕ в доминирующем режиме.
# 0 — все н.у. согласны (моностабильно); →1 — режим спорный (мода прячет раскол).
def codemulti_traces(eps):
    a = PTS[PTS['eps'] == eps]
    traces = []
    for ct in range(4):
        d = a[a['coupling_type'] == ct]
        piv = d.pivot(index='delta1', columns='delta2', values='agree_frac')
        traces.append(go.Heatmap(
            z=1 - piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
            colorscale='Magma', zmin=0, zmax=1, showscale=True,
            colorbar=dict(title='доля вне моды'),
            hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>%{z:.3f}<extra></extra>'))
    return traces


fig = go.Figure()
for idx, tr in enumerate(codemulti_traces(epsilons[0])):
    tr.visible = (idx == 0)
    fig.add_trace(tr)

frames = [go.Frame(name=str(e), data=codemulti_traces(e)) for e in epsilons]
fig.update_layout(height=800, width=800, autosize=True, margin=dict(b=140),
                  xaxis_title='δ₂', yaxis_title='δ₁',
                  title='Мультистабильность (θ): доля н.у. вне доминирующего режима')
add_eps_coupling_sliders(fig, frames, group_size=1, coupling_names=COUPLING_NAMES)
fig.show()

In [ ]:
_SPREADS = [('L_spread', 'ΔL'), ('A_spread', 'ΔA'), ('P_spread', 'ΔP')]
_REG = [(False, 'один режим', 'steelblue'), (True, 'мультистаб.', 'tomato')]


def hist_traces(agg):
    traces = []
    for si, (col, _) in enumerate(_SPREADS):
        for ct in range(4):
            d = agg[agg['coupling_type'] == ct]
            for is_multi, name, color in _REG:
                traces.append(go.Histogram(
                    x=d.loc[d['multi'] == is_multi, col], nbinsx=50,
                    marker_color=color, opacity=0.6, name=name, legendgroup=name,
                    showlegend=(si == 0 and ct == 0)))
    return traces


fig = make_subplots(rows=3, cols=4, row_titles=[lbl for _, lbl in _SPREADS],
                    column_titles=COUPLING_NAMES, horizontal_spacing=0.05, vertical_spacing=0.07)
it = iter(hist_traces(PTS[PTS['eps'] == epsilons[0]]))
for si in range(3):
    for ct in range(4):
        for _ in _REG:
            fig.add_trace(next(it), row=si + 1, col=ct + 1)
fig.update_yaxes(type='log')

frames = [go.Frame(name=str(e), data=hist_traces(PTS[PTS['eps'] == e])) for e in epsilons]
fig.update_layout(height=900, barmode='overlay', title='Разброс L, A, P vs мультистабильность (θ)')
add_eps_slider(fig, frames)
fig.show()

In [ ]:
def runs_at(eps, ct, d1, d2):
    s = sub_with_code(eps)
    return s[(s['coupling_type'] == ct)
             & np.isclose(s['delta1'], d1) & np.isclose(s['delta2'], d2)]


def counterexamples(eps, ct=0):
    d = PTS[(PTS['eps'] == eps) & (PTS['coupling_type'] == ct)]
    return (d[d['multi'] & (d['L_spread'] < 0.1)].head(3),
            d[~d['multi'] & (d['L_spread'] > 0.5)].head(3))


def show_counterexamples(eps, ct=0):
    ms, ml = counterexamples(eps, ct)
    print(f'=== ε={eps}, {COUPLING_NAMES[ct]} ===')
    print('Мультистаб. с малым ΔL (<0.1):')
    for _, r in ms.iterrows():
        codes = sorted(runs_at(eps, ct, r.delta1, r.delta2)['code'].unique())
        print(f'  δ1={r.delta1:.2f} δ2={r.delta2:.2f} ΔL={r.L_spread:.4f} коды={codes}')
    print('Один режим с большим ΔL (>0.5):')
    for _, r in ml.iterrows():
        L = runs_at(eps, ct, r.delta1, r.delta2)['L'].round(3).tolist()
        print(f'  δ1={r.delta1:.2f} δ2={r.delta2:.2f} ΔL={r.L_spread:.4f} L={L}')


show_counterexamples(epsilons[2])

### Проверка точки с маленьким $\Delta L$, но мультистабильностью

In [ ]:
def show_runs_at(eps, kind, ct=0):
    ms, ml = counterexamples(eps, ct)
    sel = ms if kind == 'multi_small' else ml
    if len(sel) == 0:
        print('нет таких точек в данных'); return
    r0 = sel.iloc[0]
    pt = runs_at(eps, ct, r0.delta1, r0.delta2)
    print(f'ε={eps} δ1={r0.delta1:.2f} δ2={r0.delta2:.2f} ({COUPLING_NAMES[ct]}), ΔL={r0.L_spread:.4f}')
    print(pt[['code', 'L', 'x0', 'y0', 'x1', 'y1', 'x2', 'y2']].round(3).to_string(index=False))


show_runs_at(epsilons[2], 'multi_small')

### Проверка точки с высоким $\Delta L$, но однорежимностью

In [ ]:
show_runs_at(epsilons[2], 'mono_large')

In [ ]:
t_trans, window = 600.0, 40.0
runs = [
    {'title': 'δ₁=-0.20 δ₂=0.12 — код 2 (0↔2), L≈0.834', 'path': '../simulation/demo/small_l_code2.csv'},
    {'title': 'δ₁=-0.20 δ₂=0.12 — код 0 (нет синхр.), L≈0.838', 'path': '../simulation/demo/small_l_code0.csv'},
    {'title': 'δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈2.1', 'path': '../simulation/demo/large_dl_high_l.csv'},
    {'title': 'δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈0.235', 'path': '../simulation/demo/large_dl_low_l.csv'},
]
colors = {'x0': '#e41a1c', 'x1': '#4daf4a', 'x2': '#377eb8'}

fig = make_subplots(rows=len(runs), cols=1, shared_xaxes=True,
                    subplot_titles=[r['title'] for r in runs])
for ri, run in enumerate(runs):
    dfr = pd.read_csv(run['path'], comment='#')
    steady = dfr[(dfr['t'] >= t_trans) & (dfr['t'] < t_trans + window)]
    for col, c in colors.items():
        fig.add_trace(go.Scatter(x=steady['t'], y=steady[col], mode='lines',
                                 line=dict(color=c, width=1), name=col,
                                 legendgroup=col, showlegend=(ri == 0)),
                      row=ri + 1, col=1)

fig.update_xaxes(title_text='t', row=len(runs), col=1)
fig.update_layout(height=1000, title='C++ vdp_sim — осцилляции в установившемся режиме')
fig.show()